In [ ]:
# import torch

# print("📦 PyTorch version:", torch.__version__)
# print("🚀 CUDA available :", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("🧠 GPU Name       :", torch.cuda.get_device_name(0))


📦 PyTorch version: 2.5.1
🚀 CUDA available : True
🧠 GPU Name       : NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [ ]:
# import faiss

# print("📦 FAISS version :", faiss.__version__)

# # Kiểm tra module FAISS-GPU có hoạt động không
# try:
#     res = faiss.StandardGpuResources()  # Nếu không lỗi là có GPU
#     print("🚀 FAISS is using GPU ✅")
# except Exception as e:
#     print("❌ FAISS is NOT using GPU:", str(e))


📦 FAISS version : 1.9.0
🚀 FAISS is using GPU ✅


In [1]:
import os
import json

import pandas as pd
from ast import literal_eval

import torch
from torch.utils.data import DataLoader
from sentence_transformers.readers import InputExample
from sentence_transformers import SentenceTransformer, models, losses, util

from mteb import MTEB
from mteb.abstasks.TaskMetadata import TaskMetadata
from mteb.abstasks.AbsTaskRetrieval import AbsTaskRetrieval

from tqdm.autonotebook import tqdm

os.environ['WANDB_DISABLED'] = 'true'

c:\Users\nguye\miniforge3\envs\legal_doc_retrieval\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## **I - Prepare Data**
---

In [ ]:
!wget -q https://huggingface.co/datasets/tmnam20/BKAI-Legal-Retrieval/resolve/main/archive.zip
!unzip -o -q archive.zip -d data

In [ ]:
corpus_data = pd.read_csv('data/corpus.csv')
train_data  = pd.read_csv('data/train.csv'    , converters={'context': literal_eval})
test_data   = pd.read_csv('data/val_split.csv', converters={'context': literal_eval})

print(f"Train data: {len(train_data)}")
print(f"Test data : {len(test_data)}")

In [ ]:
train_data['cid'] = train_data['cid'].apply(lambda x: [int(i) for i in x[1:-1].split()])
test_data['cid']  = test_data['cid'].apply(lambda x: [int(i) for i in x[1:-1].split()])

train_data.head()

In [ ]:
data    = {'train': train_data, 'test': test_data}
samples = {'train': [], 'test': []}

for subset in ['train', 'test']:
    for _, row in data[subset].iterrows():
        question = row['question']
        context  = row['context']
        for c in context:
            samples[subset].append(InputExample(texts=[question, c]))

print(f"Train size: {len(samples['train'])}")
print(f"Test size : {len(samples['test'])}")

## **II - Fine-tune Sentence Transformers model**
---

In [ ]:
os.makedirs('cache/finetune', exist_ok=True)
os.makedirs('output/finetune', exist_ok=True)

CACHE_DIR = 'cache/finetune/'
MODEL_DIR = 'output/finetune/'

BATCH_SIZE = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

In [ ]:
model_id = 'google-bert/bert-base-multilingual-cased'

word_embedding_model = models.Transformer(model_id, max_seq_length=512, cache_dir=CACHE_DIR)
pooling_model        = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False, 
    pooling_mode_max_tokens=False,
)

finetuned_model = SentenceTransformer(
    modules=[word_embedding_model, pooling_model], device=DEVICE, 
    cache_folder=CACHE_DIR
)

In [ ]:
train_dataloader = DataLoader(samples['train'], shuffle=True, batch_size=BATCH_SIZE,
                              num_workers=4, pin_memory=True, prefetch_factor=2)

print(len(train_dataloader))

In [ ]:
train_loss = losses.CachedMultipleNegativesRankingLoss(model=finetuned_model)

finetuned_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10,
    output_path=MODEL_DIR, 
    optimizer_params={'lr': 3e-5},
    show_progress_bar=True,
    use_amp=True
)

## **III - Evaluation with MTEB on the BKAI Legal Document Retrieval Dataset**
---

In [ ]:
finetuned_model = SentenceTransformer(
    MODEL_DIR, device=DEVICE, 
    model_kwargs={'torch_dtype': 'float16'}
)

In [ ]:
class BKAILegalDocRetrievalTask(AbsTaskRetrieval):
    # Metadata definition used by MTEB benchmark
    metadata = TaskMetadata(name='BKAILegalDocRetrieval',
                            description='',
                            reference='https://github.com/embeddings-benchmark/mteb/blob/main/docs/adding_a_dataset.md',
                            type='Retrieval',
                            category='s2p',
                            modalities=['text'],
                            eval_splits=['test'],
                            eval_langs=['vi'],
                            main_score='ndcg_at_10',
                            other_scores=['recall_at_10', 'precision_at_10', 'map'],
                            dataset={
                                'path'    : 'data',
                                'revision': 'd4c5a8ba10ae71224752c727094ac4c46947fa29',
                            },
                            date=('2012-01-01', '2020-01-01'),
                            form='Written',
                            domains=['Academic', 'Non-fiction'],
                            task_subtypes=['Scientific Reranking'],
                            license='cc-by-nc-4.0',
                            annotations_creators='derived',
                            dialect=[],
                            text_creation='found',
                            bibtex_citation=''
    )

    data_loaded = True # Flag

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        global corpus_data, data

        self.corpus        = {}
        self.queries       = {}
        self.relevant_docs = {}

        shared_corpus = {}
        for _, row in corpus_data.iterrows():
            cid_str                = f"c{row['cid']}"
            shared_corpus[cid_str] = {'text': row['text'], '_id': row['cid']} # Standard format for AbsTaskRetrieval

        for split in data:
            self.corpus[split]        = shared_corpus
            self.queries[split]       = {}
            self.relevant_docs[split] = {}

        for split in data:
            for i, row in data[split].iterrows():
                qid, cids = row['qid'], row['cid']
                question  = row['question']
                qid_str   = f'q{qid}'
                cids_str  = [f'c{cid}' for cid in cids]

                self.queries[split][qid_str] = question

                for cid_str in cids_str:
                    if cid_str not in self.relevant_docs[split]:
                        self.relevant_docs[split][qid_str] = {}
                    self.relevant_docs[split][qid_str][cid_str] = 1

        self.data_loaded = True

In [ ]:
custom_task = BKAILegalDocRetrievalTask()
evaluation  = MTEB(tasks=[custom_task])
evaluation.run(finetuned_model, batch_size=BATCH_SIZE)

## **IV - Retrieval**
---

In [ ]:
import faiss

In [ ]:
passages           = corpus_data['text'].tolist()
corspus_embeddings = finetuned_model.encode(
    passages, 
    batch_size=BATCH_SIZE,
    convert_to_numpy=True, 
    normalize_embeddings=True,
    show_progress_bar=True, 
    device=DEVICE
).astype(np.float32)

In [ ]:
d     = corspus_embeddings.shape[1] # 768
index = faiss.IndexFlatIP(d)

res   = faiss.StandardGpuResources()          # Use a single GPU
index = faiss.index_cpu_to_gpu(res, 0, index) # Move index to GPU
index.add(corspus_embeddings)                 # Add vectors to the index
faiss.write_index(index, "data/legal_faiss.index")

In [ ]:
def search(model, query, index, k=10):
    query_embedding = model.encode(
        query, 
        convert_to_numpy=True, 
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, indices = index.search(query_embedding, k)
    hits = [{'score': scores[0][i], 'index': indices[0][i]} for i in range(len(scores[0]))]
    return hits

In [ ]:
query = "Hợp đồng lao động là gì?"
index = faiss.read_index("data/legal_faiss.index")

hits = search(finetuned_model, query, index, k=10)
for hit in hits:
    print(f"Index: {hit['index']}, Score: {hit['score']}, Text: {passages[hit['index']]}")